# SAC arrival_v2 — history k=8 multi-seed (seed=7, TIE-BREAKER) on single_cross_s0 (1M, vanilla)

**Pre-context（commit `e00f9af` + seed=0 results 回流 2026-05-19）**：[`docs/arrival_v2_experiment_report.md`](../docs/arrival_v2_experiment_report.md) §7.8 「k=4→8 闭合 80pp gap」claim 目前是 **1 PASS + 1 PARTIAL** 的混合状态：

| §7.8 multi-seed state (2/3 anchors)      | seed | final  | peak@step    | mean39 | OOB   | Gate |
|---                                       |---:  |---:    |---           |---:    |---:   |---   |
| **s0 k=8 vanilla (§7.8 anchor)**         | 42   | **0.900** | 0.900 @ 475k | **0.636** | **0.100** | **PASS（5/5）** |
| **s0 k=8 vanilla (§7.8' 本套 seed=0)**   | 0    | 0.500  | 0.500 @ 925k | 0.260  | 0.133 | **FAIL（2/5）** |
| s0 k=4 vanilla (§7.6.4 floor)            | 42   | 0.100  | 0.367 @ 975k | 0.221  | 0.667 | FAIL |
| s0 k=4 vanilla (§7.7.1 sister)           | 0    | 0.400  | 0.533 @ 625k | 0.218  | 0.200 | FAIL |
| s1 k=4 vanilla (§7.1 upper ref)          | 42   | 0.900  | 0.900 @ 725k | 0.497  | 0.100 | PASS |

**关键 tension**：seed=42 上 k=4→8 buy 到 +80pp final + −56pp OOB；但 seed=0 上同样的切换只 buy 到 +10pp final + −7pp OOB，且最终高原在 50%。两 seed 跨 history 增益差异 **8 倍**。

**单 seed 解释不了的两种竞争 hypothesis**：

| Hypothesis | 含义 |
|---|---|
| **H1: "k=8 真的解决 information bottleneck，只是 seed=0 不幸卡 local minimum"** | seed=7 应 PASS（与 seed=42 类似），2/3 PASS 支持 H1 |
| **H2: "§7.8 anchor 本质是 seed=42 lucky strike，k=4→8 真实增益接近 seed=0 的 +10pp"** | seed=7 应 PARTIAL/REGRESS（与 seed=0 类似），2/3 FAIL 支持 H2 |

**本 notebook 任务（pure vanilla + history k=8，seed=7 第三 anchor，tie-breaker）**：

| 维度                  | §7.8 anchor (seed=42) | §7.8' (seed=0) | 本 notebook (seed=7) |
|---                    |---                    |---             |---                  |
| `--seed`              | 42                    | 0              | **7** ← 唯一变量    |
| `--history-length`    | 8                     | 8              | 8                   |
| algorithm             | vanilla SAC           | vanilla SAC    | vanilla SAC         |
| sensor layout         | s0 (DVL-only)         | s0 (DVL-only)  | s0 (DVL-only)       |
| reward                | arrival_v2            | arrival_v2     | arrival_v2          |
| flow U / target       | 1.5 / 1.5             | 1.5 / 1.5      | 1.5 / 1.5           |
| total_steps           | 1M                    | 1M             | 1M                  |
| num_envs              | 6                     | 6              | 6                   |
| benchmark             | `single_u15_cross_tgt15` | 同           | 同                   |
| obs_dim               | 96 (12×8)             | 96             | 96                  |

**Seed 选择 = 7 的理由**：与 §7 章节其它 multi-seed 套保持 prime-seed 习惯（42 / 0 / 7 是常用三元组）；避免与 §7.6 / §7.7 已用 seed 重合。

**Gate**（与 §7 / §7.6 / §7.7 / §7.8 同口径）：
- `final_success_rate ≥ 0.85`
- `last100k_mean ≥ 0.9 × peak`
- `final_oob_rate ≤ 0.10`
- `include_episode_context_obs == True`
- `timeout_bootstrap_semantics == 'terminal'`

**TIE-BREAK verdict 规则**（驱动 §7.8 主张存亡 + 决定后续 §8 P1 走向）：

| Verdict | 触发条件 | 解释 + §7.8 主张状态 |
|---|---|---|
| **TIE-BREAK-ROBUST** | seed=7 PASS（5/5）| 2/3 PASS → 支持 H1。§7.8 写「k=4→8 在 2/3 seeds 上 close gap；seed=0 fails to converge from this init」+ 加 seed=0 audit 段；可以推进 §8 P1#2 k=12 cross-seed |
| **TIE-BREAK-HIGH-VAR** | seed=7 PASS 但 \|Δfinal\| vs anchor > 0.15 | 2/3 PASS 但 noisy；§7.8 写「k=8 PASS robust but seed-noisy; recommend ≥3 seeds for thesis claim」 |
| **TIE-BREAK-WEAK** | seed=7 STRONG-PARTIAL（final ∈ [0.5, 0.85)）| 1 PASS + 2 PARTIAL → effect 部分存在；§7.8 reframe「k=8 partial mitigation, seed-sensitive 50-90% final range」；放弃 §8 P1#2 k=12 / k=16 单 seed 路径 |
| **TIE-BREAK-DEAD** | seed=7 final < 0.5 | 1/3 PASS → 强证据 H2（§7.8 anchor = seed=42 偶然命中）；§7.8 主张回退「k=4→8 effect highly seed-dependent, not a method-level gap closer」；可能转向 §8 P0 SAC variance reduction（DroQ / SimBa / N-Step）路线 |

**输出根（与 §7.8 anchor / §7.8' 互不覆盖）**：
- `experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_7/`

**总预算**：~2.5h L4（1 Colab Pro+ session）。

**风格**：训练用 `!python -u -m scripts.train_sac` 直跑（实时 stdout），与 §7.8 / §7.8' 一致。


## 0. GPU sanity


In [ ]:
!nvidia-smi | head -10


## 1. Mount Drive + cwd


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5'
%cd $REPO_DIR


## 2. Config — single phase（vanilla SAC + history k=8 + seed=7，与 §7.8 anchor 仅差一个 flag value）


In [ ]:
import json
import os
from pathlib import Path

import pandas as pd

# ==== SAC / env config (与 §7.8 anchor 严格一致，除 seed 外) ====
OBJECTIVE = 'arrival_v2'
PROBE_LAYOUT = 's0'
HISTORY_LENGTH = 8
TARGET_SPEED = 1.5
SEED = 7                                # ← 唯一与 §7.8 anchor (seed=42) 不同的值；TIE-BREAK 3rd anchor

RANDOM_STEPS = 5_000
UPDATE_AFTER = 5_000
BATCH_SIZE = 256
HIDDEN_DIM = 256
NUM_ENVS = 6
EVAL_EVERY = 25_000
EVAL_EPISODES = 30
CHECKPOINT_EVERY = 100_000
DEVICE = 'cuda'

# 显式拒绝所有 SAC 改进项 — 与 §7.8 anchor 一致
USE_ASYMMETRIC_CRITIC = False
USE_LAYERNORM = False
UPDATES_PER_STEP = 1
DROPOUT_RATE = 0.0

PASS_FINAL_SUCCESS = 0.85
PASS_LAST100_RATIO = 0.90
PASS_OOB_RATE = 0.10

# Flow file（与 §7.1 / §7.6.4 / §7.7 / §7.8 / §7.8' 严格一致）
SINGLE_FLOW = 'wake_data/wake_v8_U1p50_Re250_D12p00_dx0p60_Ti5pct_1200f_roi.npy'

# ==== Single phase — single_cross s0 k=8 seed=7 (X_*) ====
X_BENCHMARK_KEY = 'single_u15_cross_tgt15'
X_TASK_GEOMETRY = 'cross_stream'
X_FLOW_PATH = SINGLE_FLOW
X_TOTAL_STEPS = 1_000_000
X_RUN_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_7')
X_CKPT_ROOT = Path('checkpoints/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_7')
X_MANIFEST_PATH = Path(f'benchmarks/{X_BENCHMARK_KEY}.json')

# Baselines（事后对比）
X_K8_S42_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_42')   # §7.8 anchor PASS
X_K8_S0_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k8/seed_0')    # §7.8' PARTIAL (本套上一轮)
X_K4_S42_ROOT  = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_42')   # §7.6.4 floor
X_K4_S0_ROOT   = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s0_k4/seed_0')    # §7.7.1 sister
X_S1_UPPER_ROOT = Path('experiments/arrival_v2_prototype/single_u15_cross_tgt15/arrival_v2/sac_vanilla/s1_k4/seed_42')  # §7.1 upper

os.environ['PYTHONUNBUFFERED'] = '1'

print(f'PROBE_LAYOUT          = {PROBE_LAYOUT}')
print(f'OBJECTIVE             = {OBJECTIVE}')
print(f'HISTORY_LENGTH        = {HISTORY_LENGTH}')
print(f'SEED                  = {SEED}        ← TIE-BREAK 3rd anchor，与 §7.8 anchor (seed=42) / §7.8\' (seed=0) 不同')
print(f'NUM_ENVS              = {NUM_ENVS}')
print(f'USE_ASYMMETRIC_CRITIC = {USE_ASYMMETRIC_CRITIC}')
print(f'USE_LAYERNORM         = {USE_LAYERNORM}')
print(f'UPDATES_PER_STEP      = {UPDATES_PER_STEP}')
print(f'DROPOUT_RATE          = {DROPOUT_RATE}')
print()
print(f'benchmark             : {X_BENCHMARK_KEY}')
print(f'geometry              : {X_TASK_GEOMETRY}')
print(f'total_steps           : {X_TOTAL_STEPS:,}')
print(f'expected obs_dim      : 12 * {HISTORY_LENGTH} = {12 * HISTORY_LENGTH} (per train_config.txt across all baselines)')
print(f'run_root              : {X_RUN_ROOT}')
print(f'§7.8  k=8 seed=42 PASS    : {X_K8_S42_ROOT}')
print(f"§7.8' k=8 seed=0  PARTIAL : {X_K8_S0_ROOT}")
print(f'§7.6.4 k=4 seed=42 floor  : {X_K4_S42_ROOT}')
print(f'§7.7.1 k=4 seed=0  sister : {X_K4_S0_ROOT}')
print(f'§7.1  s1 k=4 upper ref    : {X_S1_UPPER_ROOT}')


## 3. Preflight — flow / arrival_v2 candidate gate / reward unit tests / manifest / baseline 就位


In [ ]:
# Flow file
fp = Path(X_FLOW_PATH)
if not fp.exists():
    raise FileNotFoundError(f'missing flow file: {fp}')
print(f'[OK] flow file: {fp}  ({fp.stat().st_size / 1e6:.1f} MB)')


In [ ]:
!python -u -m scripts.validate_arrival_v2_candidate


In [ ]:
!python -u -m pytest tests/test_reward_objective.py -q


In [ ]:
if not X_MANIFEST_PATH.exists():
    !python -u -m scripts.generate_standard_benchmarks --benchmarks {X_BENCHMARK_KEY} --episodes {EVAL_EPISODES}
if not X_MANIFEST_PATH.exists():
    raise FileNotFoundError(f'manifest not generated: {X_MANIFEST_PATH}')
print(f'[OK] manifest ready: {X_MANIFEST_PATH}')


In [ ]:
# 检查 5 个对比 baseline 是否就位
for label, root, ref_final, ref_oob in [
    ('§7.8  k8 seed=42 PASS    ', X_K8_S42_ROOT, 0.900, 0.100),
    ("§7.8' k8 seed=0  PARTIAL ", X_K8_S0_ROOT,  0.500, 0.133),
    ('§7.6.4 k4 seed=42 floor   ', X_K4_S42_ROOT, 0.100, 0.667),
    ('§7.7.1 k4 seed=0  sister  ', X_K4_S0_ROOT,  0.400, 0.200),
    ('§7.1  s1 k=4 upper ref    ', X_S1_UPPER_ROOT, 0.900, 0.100),
]:
    fp = root / 'results' / 'final_eval.json'
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        oob = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        match = '✓' if abs(f - ref_final) < 0.05 and abs(oob - ref_oob) < 0.05 else '✗ mismatch'
        print(f'[OK] {label}: final={f:.4f}  oob={oob:.4f}  counts={c}  {match}')
    else:
        print(f'[WARN] {label}: {fp} 不存在 (后续 §5 diff 会回退到 report 转载值)')


## 4. Train — single_cross_s0 + history k=8 + seed=7 (1.0M, skip/resume)


In [ ]:
x_state_path = X_RUN_ROOT / 'trainer_state.json'
if x_state_path.exists():
    x_state = json.loads(x_state_path.read_text(encoding='utf-8'))
    x_current_step = int(x_state.get('env_step', 0))
else:
    x_current_step = 0
print(f'[state] X env_step = {x_current_step:,} / target {X_TOTAL_STEPS:,}')

if x_current_step >= X_TOTAL_STEPS:
    print(f'[skip] X already trained to {x_current_step:,} >= {X_TOTAL_STEPS:,}')
elif x_current_step > 0:
    print(f'[resume] X continuing from {x_current_step:,} -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --resume {str(X_RUN_ROOT)} \
        --total-steps {X_TOTAL_STEPS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --device {DEVICE}
else:
    print(f'[train] X fresh start -> {X_TOTAL_STEPS:,}')
    !python -u -m scripts.train_sac \
        --flow {X_FLOW_PATH} \
        --task-geometry {X_TASK_GEOMETRY} \
        --target-speed {TARGET_SPEED} \
        --objective {OBJECTIVE} \
        --probe-layout {PROBE_LAYOUT} \
        --history-length {HISTORY_LENGTH} \
        --total-steps {X_TOTAL_STEPS} \
        --random-steps {RANDOM_STEPS} \
        --update-after {UPDATE_AFTER} \
        --batch-size {BATCH_SIZE} \
        --hidden-dim {HIDDEN_DIM} \
        --num-envs {NUM_ENVS} \
        --eval-every {EVAL_EVERY} \
        --eval-episodes {EVAL_EPISODES} \
        --checkpoint-every {CHECKPOINT_EVERY} \
        --eval-manifest {str(X_MANIFEST_PATH)} \
        --seed {SEED} \
        --device {DEVICE} \
        --save-dir {str(X_RUN_ROOT)} \
        --checkpoint-dir {str(X_CKPT_ROOT)}


## 5. Summary + gate


In [ ]:
def summarize_phase(run_root: Path, total_steps: int, label: str, gate_filename: str):
    eval_log_path = run_root / 'results' / 'eval_log.csv'
    final_eval_path = run_root / 'results' / 'final_eval.json'
    trainer_state_path = run_root / 'trainer_state.json'
    train_config_path = run_root / 'results' / 'train_config.txt'

    if not eval_log_path.exists():
        raise FileNotFoundError(f'missing eval log: {eval_log_path}')
    if not final_eval_path.exists():
        raise FileNotFoundError(f'missing final eval: {final_eval_path}')

    df = pd.read_csv(eval_log_path)
    final_eval = json.loads(final_eval_path.read_text(encoding='utf-8'))
    trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8')) if trainer_state_path.exists() else {}

    history_from_config = 'NA'
    obs_dim_from_config = 'NA'
    if train_config_path.exists():
        for ln in train_config_path.read_text(encoding='utf-8').splitlines():
            s = ln.strip()
            if s.startswith('history_length='):
                history_from_config = s.split('=', 1)[1]
            elif s.startswith('obs_dim='):
                obs_dim_from_config = s.split('=', 1)[1]

    peak_success = float(df['eval_success_rate'].max()) if len(df) else 0.0
    peak_step = int(df.loc[df['eval_success_rate'].idxmax(), 'env_step']) if len(df) else 0
    last100 = df[df['env_step'] >= total_steps - 100_000].copy()
    last100_mean = float(last100['eval_success_rate'].mean()) if len(last100) else 0.0
    mean_all = float(df['eval_success_rate'].mean()) if len(df) else 0.0
    n_evals_with_success = int((df['eval_success_rate'] > 0).sum()) if len(df) else 0
    final_success = float(final_eval['eval_success_rate'])
    counts = final_eval.get('eval_termination_counts', {})
    num_eps = float(final_eval.get('num_eval_episodes', EVAL_EPISODES))
    oob_rate = float(counts.get('out_of_bounds', 0)) / max(num_eps, 1.0)

    print('=' * 100)
    print(f'{label}  (arrival_v2 / s0 / k=8 / seed={SEED} / vanilla / {total_steps:,} steps)')
    print('-' * 100)
    print(f"  final_success_rate    : {final_success:.4f}   gate >= {PASS_FINAL_SUCCESS:.2f}")
    print(f"  peak_success_rate     : {peak_success:.4f}   @ {peak_step:,}")
    print(f"  last100k_mean_success : {last100_mean:.4f}   gate >= {PASS_LAST100_RATIO * peak_success:.4f}")
    print(f"  full-traj mean        : {mean_all:.4f}   (39 evals)")
    print(f"  n_evals_with_success  : {n_evals_with_success} / {len(df)}")
    print(f"  final_oob_rate        : {oob_rate:.4f}   gate <= {PASS_OOB_RATE:.2f}")
    print(f"  obs_dim               : {obs_dim_from_config}   (expect 12*8=96)")
    print(f"  history_length        : {history_from_config}   (from train_config.txt)")
    print(f"  context_obs           : {trainer_state.get('include_episode_context_obs', 'NA')}")
    print(f"  timeout_bootstrap     : {trainer_state.get('timeout_bootstrap_semantics', 'NA')}")
    print(f"  termination           : {counts}")
    print('=' * 100)

    if len(df):
        print()
        print('[last 16 eval rows]')
        cols = ['env_step', 'eval_success_rate', 'eval_return', 'eval_safety_cost',
                'eval_time_s', 'eval_progress_ratio']
        available = [c for c in cols if c in df.columns]
        print(df[available].tail(16).to_string(index=False))

    checks = [
        ('final success >= 0.85', final_success >= PASS_FINAL_SUCCESS, f'{final_success:.4f}'),
        ('last100k mean >= 0.9 * peak', last100_mean >= PASS_LAST100_RATIO * peak_success,
         f'{last100_mean:.4f} / peak={peak_success:.4f}'),
        ('final OOB rate <= 0.10', oob_rate <= PASS_OOB_RATE, f'{oob_rate:.4f}'),
        ('arrival_v2 context obs enabled',
         trainer_state.get('include_episode_context_obs') is True,
         str(trainer_state.get('include_episode_context_obs'))),
        ('arrival_v2 timeout terminal semantics',
         trainer_state.get('timeout_bootstrap_semantics') == 'terminal',
         str(trainer_state.get('timeout_bootstrap_semantics'))),
    ]

    print()
    print('=' * 96)
    print(f"{'check':<48}{'pass':>8}{'detail':>40}")
    print('-' * 96)
    all_pass = True
    for name, ok, detail in checks:
        mark = 'PASS' if ok else 'FAIL'
        if not ok:
            all_pass = False
        print(f'{name:<48}{mark:>8}{detail:>40}')
    print('=' * 96)

    summary = {
        'phase': label,
        'objective': OBJECTIVE,
        'probe_layout': PROBE_LAYOUT,
        'history_length': int(history_from_config) if history_from_config != 'NA' else HISTORY_LENGTH,
        'seed': SEED,
        'total_steps': total_steps,
        'algorithm': 'sac_vanilla',
        'final_success_rate': final_success,
        'peak_success_rate': peak_success,
        'peak_step': peak_step,
        'last100k_mean_success': last100_mean,
        'mean_success_full_trajectory': mean_all,
        'n_evals_with_success': n_evals_with_success,
        'n_evals_total': len(df),
        'final_oob_rate': oob_rate,
        'termination_counts': counts,
        'all_pass': bool(all_pass),
        'checks': [{'name': n, 'ok': bool(ok), 'detail': d} for n, ok, d in checks],
    }
    out_path = run_root / 'results' / gate_filename
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f'[saved] {out_path}')
    return summary

x_summary = summarize_phase(
    X_RUN_ROOT,
    X_TOTAL_STEPS,
    'SINGLE_CROSS_S0_K8_SEED7',
    'single_cross_s0_k8_seed7_gate_summary.json',
)
X_PASS = x_summary['all_pass']
print()
print(f'X_PASS = {X_PASS}')


## 6. TIE-BREAK verdict — k=8 seed=7 vs §7.8 anchor (seed=42 PASS) + §7.8' (seed=0 PARTIAL) + 全表 7-run 对照


In [ ]:
def read_baseline(root: Path, ref_final: float, ref_oob: float, ref_mean: float = None):
    fp = root / 'results' / 'final_eval.json'
    log = root / 'results' / 'eval_log.csv'
    out = {'final': ref_final, 'oob': ref_oob, 'mean': ref_mean, 'peak': None, 'peak_step': None, 'source': 'report'}
    if fp.exists():
        d = json.loads(fp.read_text(encoding='utf-8'))
        f = float(d['eval_success_rate'])
        c = d.get('eval_termination_counts', {})
        n = float(d.get('num_eval_episodes', EVAL_EPISODES))
        out['final'] = f
        out['oob'] = float(c.get('out_of_bounds', 0)) / max(n, 1.0)
        out['source'] = 'on-disk'
    if log.exists():
        dlog = pd.read_csv(log)
        out['mean'] = float(dlog['eval_success_rate'].mean())
        out['peak'] = float(dlog['eval_success_rate'].max())
        out['peak_step'] = int(dlog.loc[dlog['eval_success_rate'].idxmax(), 'env_step'])
    return out

print('=' * 100)
print('SINGLE_CROSS — k=8 multi-seed TIE-BREAK verdict (seed=7 vs seed=42 PASS + seed=0 PARTIAL)')
print('-' * 100)

# 本 run (k=8 seed=7)
k8s7_final = float(x_summary['final_success_rate'])
k8s7_oob   = float(x_summary['final_oob_rate'])
k8s7_peak  = float(x_summary['peak_success_rate'])
k8s7_mean  = float(x_summary.get('mean_success_full_trajectory', 0.0))
k8s7_nsucc = int(x_summary.get('n_evals_with_success', 0))
k8s7_ntot  = int(x_summary.get('n_evals_total', 0))

# 5 个 baseline（report-fallback 用 §7.8 / §7.8' / §7.6.4 / §7.7.1 / §7.1 转载值）
b_k8_s42 = read_baseline(X_K8_S42_ROOT,   0.900, 0.100, 0.636)   # §7.8 anchor PASS
b_k8_s0  = read_baseline(X_K8_S0_ROOT,    0.500, 0.133, 0.260)   # §7.8' PARTIAL
b_k4_s42 = read_baseline(X_K4_S42_ROOT,   0.100, 0.667, 0.221)   # §7.6.4 floor
b_k4_s0  = read_baseline(X_K4_S0_ROOT,    0.400, 0.200, 0.218)   # §7.7.1 sister
b_s1     = read_baseline(X_S1_UPPER_ROOT, 0.900, 0.100, 0.497)   # §7.1 upper

print()
print(f'{"config":<48}{"final":>10}{"mean":>10}{"oob":>10}{"peak":>10}{"source":>12}')
print('-' * 100)
print(f'{"vanilla s1_k4 s42 (§7.1 upper ref)":<48}{b_s1["final"]:>10.4f}{(b_s1["mean"] or float("nan")):>10.4f}{b_s1["oob"]:>10.4f}{(b_s1["peak"] or float("nan")):>10.4f}{b_s1["source"]:>12}')
print(f'{"vanilla s0_k4 s42 (§7.6.4 FAIL floor)":<48}{b_k4_s42["final"]:>10.4f}{(b_k4_s42["mean"] or float("nan")):>10.4f}{b_k4_s42["oob"]:>10.4f}{(b_k4_s42["peak"] or float("nan")):>10.4f}{b_k4_s42["source"]:>12}')
# Pre-format apostrophe-containing label (avoid backslash inside f-string {...} expression)
label_k8_s0 = "vanilla s0_k8 s0  (§7.8' PARTIAL)"
print(f'{"vanilla s0_k4 s0  (§7.7.1 sister)":<48}{b_k4_s0["final"]:>10.4f}{(b_k4_s0["mean"] or float("nan")):>10.4f}{b_k4_s0["oob"]:>10.4f}{(b_k4_s0["peak"] or float("nan")):>10.4f}{b_k4_s0["source"]:>12}')
print(f'{"vanilla s0_k8 s42 (§7.8 anchor PASS)":<48}{b_k8_s42["final"]:>10.4f}{(b_k8_s42["mean"] or float("nan")):>10.4f}{b_k8_s42["oob"]:>10.4f}{(b_k8_s42["peak"] or float("nan")):>10.4f}{b_k8_s42["source"]:>12}')
print(f'{label_k8_s0:<48}{b_k8_s0["final"]:>10.4f}{(b_k8_s0["mean"] or float("nan")):>10.4f}{b_k8_s0["oob"]:>10.4f}{(b_k8_s0["peak"] or float("nan")):>10.4f}{b_k8_s0["source"]:>12}')
print(f'{"vanilla s0_k8 s7  (THIS RUN, TIE-BREAK)":<48}{k8s7_final:>10.4f}{k8s7_mean:>10.4f}{k8s7_oob:>10.4f}{k8s7_peak:>10.4f}{"on-disk":>12}')
print('=' * 100)

# Δ 行
delta_final_vs_s42 = k8s7_final - b_k8_s42['final']
delta_mean_vs_s42  = k8s7_mean  - (b_k8_s42['mean'] or 0.0)
delta_oob_vs_s42   = k8s7_oob   - b_k8_s42['oob']
delta_final_vs_s0  = k8s7_final - b_k8_s0['final']
delta_mean_vs_s0   = k8s7_mean  - (b_k8_s0['mean'] or 0.0)

print()
print(f'{"contrast":<60}{"Δ final":>12}{"Δ mean":>12}{"Δ oob":>12}')
print('-' * 100)
print(f'{"k=8 seed=7 vs k=8 seed=42 (vs PASS anchor)":<60}'
      f'{delta_final_vs_s42:>+12.4f}{delta_mean_vs_s42:>+12.4f}{delta_oob_vs_s42:>+12.4f}')
print(f'{"k=8 seed=7 vs k=8 seed=0 (vs PARTIAL sister)":<60}'
      f'{delta_final_vs_s0:>+12.4f}{delta_mean_vs_s0:>+12.4f}{(k8s7_oob - b_k8_s0["oob"]):>+12.4f}')
print(f'{"k=8 seed=7 vs s1_k4 seed=42 (gap-to-upper)":<60}'
      f'{k8s7_final - b_s1["final"]:>+12.4f}'
      f'{(k8s7_mean - (b_s1["mean"] or 0)):>+12.4f}'
      f'{k8s7_oob - b_s1["oob"]:>+12.4f}')
print('=' * 100)
print()
print(f'k=8 seed=7 evals_with_success: {k8s7_nsucc} / {k8s7_ntot}  '
      f'(seed=42 was 37/39 PASS, seed=0 was 32/39 PARTIAL, k=4 seed=42 was 35/39 FAIL, k=4 seed=0 was 34/39 FAIL)')

# TIE-BREAK verdict — 4 档
THRESH_FINAL = 0.15   # paired |Δfinal| 容忍度 (vs anchor)
abs_delta_vs_s42 = abs(delta_final_vs_s42)
seed7_pass = (k8s7_final >= PASS_FINAL_SUCCESS) and (k8s7_oob <= PASS_OOB_RATE)
seed7_strong_partial = (k8s7_final >= 0.50) and (k8s7_final < PASS_FINAL_SUCCESS)
seed7_regress = (k8s7_final < 0.50)

# 3-seed aggregate state
seed42_pass = True   # §7.8 anchor
seed0_pass  = False  # §7.8' PARTIAL
n_pass_total = int(seed42_pass) + int(seed0_pass) + int(seed7_pass)

if seed7_pass and abs_delta_vs_s42 <= THRESH_FINAL:
    verdict = (
        f'TIE-BREAK-ROBUST — seed=7 PASS 且 |Δfinal vs anchor|={abs_delta_vs_s42:.3f}≤0.15；'
        f'3-seed state: {n_pass_total}/3 PASS (seed=42 PASS + seed=7 PASS + seed=0 PARTIAL)；'
        '支持 H1。§7.8 主张升格为 "k=4→8 closes 80pp gap in 2/3 seeds; seed=0 hits local minimum, '
        'requires init audit"。下一步：§8 P1#2 k=12 seed=0 sister 验 monotonicity 跨 seed 鲁棒性'
    )
elif seed7_pass and abs_delta_vs_s42 > THRESH_FINAL:
    verdict = (
        f'TIE-BREAK-HIGH-VAR — seed=7 PASS 但 |Δfinal vs anchor|={abs_delta_vs_s42:.3f}>0.15；'
        f'3-seed state: {n_pass_total}/3 PASS but high variance；'
        '§7.8 写 "k=8 PASS robust modulo high seed variance; thesis claim requires ≥4 seeds"；'
        '下一步：考虑 §8 P0 SAC variance reduction (DroQ / SimBa / N-Step) 改善 stability'
    )
elif seed7_strong_partial:
    verdict = (
        f'TIE-BREAK-WEAK — seed=7 STRONG-PARTIAL (final={k8s7_final:.3f}∈[0.5,0.85))；'
        '3-seed state: 1 PASS + 2 PARTIAL；'
        'effect 部分存在但 seed-sensitive；§7.8 reframe "k=8 buys partial mitigation: '
        'final 50-90% range with strong seed dependence"；'
        '放弃 §8 P1#2 k=12 / k=16 单 seed 路径；考虑直接转向 §8 P0 SAC variance reduction'
    )
else:  # seed7_regress
    verdict = (
        f'TIE-BREAK-DEAD — seed=7 REGRESS (final={k8s7_final:.3f}<0.50)；'
        '3-seed state: 1/3 PASS only；'
        '强证据 H2：§7.8 anchor 本质 seed=42 lucky strike，k=4→8 真实增益接近 +10pp 而非 +80pp；'
        '§7.8 主张回退「k=4→8 effect highly seed-dependent, not a method-level gap closer」；'
        '必须转向 §8 P0 SAC variance reduction (DroQ / SimBa / N-Step) 路线，'
        '或重新审视 actor temporal info hypothesis（可能是 critic estimation noise 同样大）'
    )

print()
print(f'>>> verdict: {verdict}')

# 落盘 tie-break summary
tiebreak_out = {
    'experiment': 'arrival_v2_s0_cross_k8_seed7_tiebreak',
    'seed': SEED,
    'benchmark': X_BENCHMARK_KEY,
    'probe_layout': PROBE_LAYOUT,
    'history_length': HISTORY_LENGTH,
    'total_steps': X_TOTAL_STEPS,
    'algorithm': 'sac_vanilla',
    'cli_diff_vs_§7.8_anchor': '--seed 42 → --seed 7',
    'results': {
        'k8_s7_thisrun': {
            'final': k8s7_final, 'mean': k8s7_mean, 'oob': k8s7_oob, 'peak': k8s7_peak,
            'n_evals_with_success': k8s7_nsucc, 'n_evals_total': k8s7_ntot,
        },
        'k8_s42_anchor_§7.8_PASS': b_k8_s42,
        "k8_s0_§7.8'_PARTIAL": b_k8_s0,
        '§7.6.4_k4_s42_floor': b_k4_s42,
        '§7.7.1_k4_s0_sister': b_k4_s0,
        '§7.1_s1_k4_s42_upper': b_s1,
    },
    'delta_vs_§7.8_anchor_seed42': {
        'final_pp': round(delta_final_vs_s42 * 100, 2),
        'mean_pp':  round(delta_mean_vs_s42  * 100, 2),
        'oob_pp':   round(delta_oob_vs_s42   * 100, 2),
    },
    "delta_vs_§7.8'_seed0": {
        'final_pp': round(delta_final_vs_s0 * 100, 2),
        'mean_pp':  round(delta_mean_vs_s0  * 100, 2),
        'oob_pp':   round((k8s7_oob - b_k8_s0['oob']) * 100, 2),
    },
    'multi_seed_state': {
        'seed42': 'PASS',
        'seed0': 'PARTIAL',
        'seed7': 'PASS' if seed7_pass else ('STRONG-PARTIAL' if seed7_strong_partial else 'REGRESS'),
        'n_pass_total': n_pass_total,
    },
    'all_pass': bool(x_summary['all_pass']),
    'verdict': verdict,
}
out_dir = Path('experiments/arrival_v2_prototype/s0_cross_k8_seed7_summary')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'combined_gate_summary.json'
out_path.write_text(json.dumps(tiebreak_out, indent=2), encoding='utf-8')
print(f'\n[saved] {out_path}')
